# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates idempotent UPSERT/DELETE logic on a simulated stateful Iceberg table using `MERGE INTO`.

In [ ]:
import os
from pyspark.sql import SparkSession

os.environ['SPARK_VERSION'] = '3.3'

spark = SparkSession.builder \
    .appName("Day07_Incremental_Upsert") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.apache.iceberg:iceberg-aws-bundle:1.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("SparkSession initialized for Iceberg MERGE operations.")

In [ ]:
# Setup Initial Target Table (Silver)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.silver_db")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.silver_db.patients (
    patient_id INT,
    name STRING,
    status STRING,
    updated_at TIMESTAMP,
    cdc_op STRING
)
USING iceberg
""")

# Insert initial state
spark.sql("""
INSERT INTO local.silver_db.patients VALUES
(1, 'Alice', 'Active', timestamp '2023-01-01 10:00:00', 'c'),
(2, 'Bob', 'Active', timestamp '2023-01-01 10:00:00', 'c')
""")

print("Initial State in Silver Table:")
spark.sql("SELECT * FROM local.silver_db.patients ORDER BY patient_id").show()

In [ ]:
# Simulate CDC Incremental Batch
# Includes:
# - Update to Alice (late arriving, older timestamp, should be ignored!)
# - Update to Alice (newer timestamp, should apply)
# - Delete Bob
# - Insert Charlie

cdc_data = [
    (1, 'Alice', 'Inactive', '2023-01-01 09:00:00', 'u'), # Out of order, should ignore
    (1, 'Alice', 'Discharged', '2023-01-02 11:00:00', 'u'), # Valid update
    (2, 'Bob', 'Active', '2023-01-02 10:00:00', 'd'),      # Delete
    (3, 'Charlie', 'Active', '2023-01-02 10:00:00', 'c')   # Insert
]
df_cdc = spark.createDataFrame(cdc_data, ["patient_id", "name", "status", "updated_at", "cdc_op"])
df_cdc = df_cdc.withColumn("updated_at", df_cdc["updated_at"].cast("timestamp"))

df_cdc.createOrReplaceTempView("incremental_updates")
print("Incoming CDC Batch:")
df_cdc.show()

In [ ]:
# Execute MERGE INTO
merge_query = """
MERGE INTO local.silver_db.patients t
USING (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER(PARTITION BY patient_id ORDER BY updated_at DESC) as rn 
        FROM incremental_updates
    ) WHERE rn = 1
) s
ON t.patient_id = s.patient_id
WHEN MATCHED AND s.cdc_op = 'd' THEN DELETE
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED AND s.cdc_op != 'd' THEN INSERT *
"""

spark.sql(merge_query)
print("MERGE applied.")

In [ ]:
# Verify Final State
print("Final State in Silver Table:")
spark.sql("SELECT * FROM local.silver_db.patients ORDER BY patient_id").show()